[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/optics/crystal_diffraction/crystal_diffraction.ipynb)

# Crystal Deformation and Diffraction

This example computes how a crystal's diffraction pattern shifts and shears when the crystal deforms, modelling the scattered light as an electromagnetic field measured by an observer at rest with the crystal. When a crystal stretches, its atoms move apart while its diffraction peaks move closer together so that waves scattered from every atom continue to arrive in phase at each peak. Circularly polarized light turns its electric and magnetic fields steadily in its wavefront plane as the wave advances, encoding each atom's scattering phase as a spatial field rotation whose summed energy density gives the observed brightness.

In spacetime geometric algebra, atom events are intrinsic antivectors and wavevectors are extrinsic vectors whose pairing `wavevector & event` gives the wave phase directly. A crystal deformation maps events forward while leaving the observer's time fixed, and its adjugate pulls wavevectors back by incidence alone through `deformation.adjugate()(k) & x == k & deformation(x)`, so `deformation.adjugate().solve(lattice)` carries every reciprocal peak forward without invoking a metric. Turning the circularly polarized null wave in its wavefront plane via `(wavefront_plane * (phases / 2)).exp() >> incident` scatters each atom's contribution with helicity set by the bivector sense, and the bilinear form `0.5 * (t & (Field >> time))` reads the observer's positive energy density; see the spatial-phasor comparison twin in [crystal_diffraction_space.ipynb](crystal_diffraction_space.ipynb).

In matrix crystallography ([ORNL's diffraction primer](https://web.archive.org/web/20250309102157id_/https://neutrons.ornl.gov/sites/default/files/NX_Miller_2018.pdf)) and Riemann–Silberstein complex-vector optics, a spatial deformation matrix $F$ maps atom coordinates by $\mathbf{x}' = F\mathbf{x}$ while reciprocal lattice vectors transform contragrediently by $\mathbf{g}' = F^{-\mathsf{T}}\mathbf{g}$ so that $\mathbf{g}'^{\mathsf{T}}\mathbf{x}' = \mathbf{g}^{\mathsf{T}}\mathbf{x}$ (or $\eta^{-1}F^{-\mathsf{T}}\eta\,k$ when raised and lowered with the Minkowski metric $\eta$). Circularly polarized scattering then combines separate electric and magnetic 3-vectors into a complex vector $\mathbf{F} = \mathbf{E} + i\mathbf{B}$ advanced by unit-imaginary phase factors $\mathbf{F}_0\,e^{i\,\mathbf{q}\cdot\mathbf{x}'_j}$ and read out via the Hermitian inner product $\tfrac{1}{2}\,\mathbf{F}^*\!\cdot\mathbf{F} = \tfrac{1}{2}(\mathbf{E}^2 + \mathbf{B}^2)$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
import numpy as np

from examples.optics.crystal_diffraction import render
from examples.util.animation import save_figure
from numga import NumpyContext
from numga.algebras import STA

# Spacetime algebra with signature x+y+z+t-.
ga = STA
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                         # Scalar+
Wavevector = ga.gatype.vector()                                                     # Vector+
Event = ga.gatype.antivector()                                                      # Antivector-
Field = ga.gatype.bivector()                                                        # Bivector+
Deformation = ga.gatype((Event, Event))                                             # Antivector- <- Antivector-
# Circularly polarized light propagating along z turns its field in the xy wavefront plane.
wavefront_plane = mv.xy                                                             # [] Field+
# Rest-frame time direction pairing to +1 with the spatial slice `mv.t` under `t-` signature.
time = -mv.t.dual()                                                                 # [] Event-
# Sandwiching `time` by a field gives twice its energy current, read against the observer's `mv.t` slice.
energy_density = 0.5 * (mv.t & (Field >> time))                                     # [] Scalar+ <- (Field+, Field+)

## Lattice deformation and its adjugate

Atoms in the undeformed crystal sit on a finite square grid at observer time zero, and its reciprocal lattice consists of the wavevectors whose pairing with every atom is an integer multiple of a full turn. Applying `layer_deformation` stretches, shears, and rotates the atom events in the layer plane while leaving observer time untouched, and solving `adjugates[:, None].solve(lattice)` carries each reciprocal wavevector to the unique moved peak that preserves every atom's phase.

In [ ]:
# Unstrained square layer, uniaxial x-stretch, and combined stretch, shear, and rotation.
n_shells = 3
n_orders = 3
n_atoms = (2 * n_shells + 1) ** 2
n_reflections = (2 * n_orders + 1) ** 2
spacing = 1.0
stretches = np.array([1.0, 1.7, 1.7])
shears = np.array([0.0, 0.0, 0.6])
angles = np.array([0.0, 0.0, np.pi / 9])
n_cases = len(stretches)

In [ ]:
shell_indices = np.arange(-n_shells, n_shells + 1)
atoms = (mv.x * shell_indices[None, :] + mv.y * shell_indices[:, None]).reshape(-1).dual() * spacing  # [n_atoms] Event-
order_indices = np.arange(-n_orders, n_orders + 1)
lattice = (
    (mv.x * order_indices[None, :] + mv.y * order_indices[:, None]).reshape(-1) * (2 * np.pi / spacing)
)                                                                                   # [n_reflections] Wavevector+


def layer_deformation(stretches: np.ndarray, shears: np.ndarray, angles: np.ndarray) -> Deformation:
    """Planar stretches along x, shears from y into x, and xy rotations form spacetime event maps (`[...] Event- <- Event-`) that leave observer time fixed."""
    local_map = (
        Event
        + (stretches - 1) * mv.x.dual() * (mv.x & Event)
        + shears * mv.x.dual() * (mv.y & Event)
    )                                                                               # [...] Event- <- Event-
    # A positive angle rotates x toward y in the layer plane.
    rotors = (mv.xy * (-angles / 2)).exp()                                          # [...] Rotor+
    return rotors >> local_map                                                      # [...] Event- <- Event-


deformations = layer_deformation(stretches, shears, angles)                         # [n_cases] Event- <- Event-
positions = deformations[:, None](atoms)                                            # [n_cases, n_atoms] Event-

# The adjugate pulls phase measurements back; solving through it carries reciprocal peaks forward.
adjugates = deformations.adjugate()                                                 # [n_cases] Wavevector+ <- Wavevector+
reciprocal = adjugates[:, None].solve(lattice)                                      # [n_cases, n_reflections] Wavevector+

## Scattered field and observer energy density

A circularly polarized plane wave travelling along the normal to the layer is represented by the null bivector `(t - z) ^ x`, whose electric and magnetic parts rotate in `wavefront_plane` as the phase advances. At each sampled momentum transfer, `scattered_field` rotates `incident` by half the transfer-event pairing `transfers & positions` at each atom and averages the scattered fields, so constructive interference at a reciprocal lattice peak reproduces the unit energy density of the incident wave while off-peak contributions cancel.

In [ ]:
half_width = 10.0
n_pixels = 101
samples = np.linspace(-half_width, half_width, n_pixels)
transfers = mv.x * samples[None, :] + mv.y * samples[:, None]                       # [n_pixels, n_pixels] Wavevector+
# Null wave propagating along +z with initial electric field along x.
wavevector = mv.t - mv.z                                                            # [] Wavevector+
incident = wavevector ^ mv.x                                                        # [] Field+

In [ ]:
def scattered_field(positions: Event, transfers: Wavevector, incident: Field) -> Field:
    """Coherent scattering from equal atoms (`[..., n_atoms] Event-`) across `transfers` (`[..., n_rows, n_columns] Wavevector+`) sums the phase-turned `incident` wave into `[..., n_rows, n_columns] Field+`."""
    n_atoms = positions.shape[-1]
    # Elastic momentum transfers have no part along observer time, so event times drop out of the phase.
    phases = transfers[..., None] & positions[..., None, None, :]                   # [..., n_atoms] Scalar+
    # Advancing a circularly polarized wave rotates its field in the wavefront plane with sense set by helicity.
    phase_rotors = (wavefront_plane * (phases / 2)).exp()                           # [..., n_atoms] Rotor+
    atom_waves = phase_rotors >> incident                                           # [..., n_atoms] Field+
    return atom_waves.sum(axis=-1) / n_atoms                                        # [...] Field+


fields = scattered_field(positions, transfers, incident)                            # [n_cases, n_pixels, n_pixels] Field+
brightness = energy_density(fields, fields)                                         # [n_cases, n_pixels, n_pixels] Scalar+
save_figure(render.comparison_figure(positions, reciprocal, transfers, brightness), "crystal_diffraction");

The top row shows the atom positions for the unstrained square layer, the x-stretched layer, and the stretched, sheared, and rotated layer. The bottom row shows the observer's scattered energy density across the momentum-transfer plane on a logarithmic colour scale, with cyan rings marking the reciprocal lattice wavevectors carried by `adjugates[:, None].solve(lattice)`.

In [ ]:
# checks
moved_phases = reciprocal[:, :, None] & positions[:, None, :]                       # [n_cases, n_reflections, n_atoms] Scalar+
rest_phases = lattice[:, None] & atoms[None, :]                                     # [n_reflections, n_atoms] Scalar+
assert ((moved_phases - rest_phases).abs() < 1e-6).all()

quarter_rotor = (wavefront_plane * (np.pi / 4)).exp()                               # [] Rotor+
turned_incident = quarter_rotor >> incident                                         # [] Field+
quarter_diff = turned_incident - (mv.y ^ wavevector)                                # [] Field+
assert incident.squared().abs() < 1e-6
assert quarter_diff.scalar_norm_squared() + quarter_diff.dual().scalar_norm_squared() < 1e-6

peak_fields = scattered_field(positions, reciprocal[:, None, :], incident)          # [n_cases, 1, n_reflections] Field+
peak_brightness = energy_density(peak_fields, peak_fields)                          # [n_cases, 1, n_reflections] Scalar+
assert ((peak_brightness - 1).abs() < 1e-5).all()
assert (brightness >= 0).all()